## Data Labeling

In [2]:
import os 
import sys 
from pathlib import Path
root_path = os.path.abspath(Path(os.getcwd()).parent)
sys.path.append(root_path)

In [3]:
import pandas as pd 
from pathlib import Path
from preprocessing import config as C
from tqdm.auto import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed
import mediapipe as mp
import cv2
import json

c:\Users\hassa\AppData\Local\pypoetry\Cache\virtualenvs\virtual-fashion-fitting-6Kc5oN_n-py3.11\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
deep_fashion_annotation = C.CLEAN_DIR / "DeepFashion" / "annotation" / "cleaned_metadata.csv"
fashion_gen_annotation = C.CLEAN_DIR / "Fashion-gen" / "annotation" / "cleaned_metadata.csv"

deep_fashion_df = pd.read_csv(deep_fashion_annotation)
fashion_gen_df = pd.read_csv(fashion_gen_annotation)

def categories_to_ids(df: pd.DataFrame, data_path: Path):
    categories = sorted(df['category'].unique())
    cat2id = {
        cat: idx for idx,cat in enumerate(categories)
    }

    df["label_id"] = df["category"].map(cat2id)
    df.to_csv(data_path, index=False)
    
categories_to_ids(deep_fashion_df, deep_fashion_annotation)
categories_to_ids(fashion_gen_df, fashion_gen_annotation)

### Pose Estimation


- iterates over every cleaned image in deep fashion images dataset 
- runs mediapipe to extract keypoints 
- saves a json alongside each image

In [5]:
def process_single_image(img_path_str, base_str, out_root_str):
    img_path = Path(img_path_str)
    base = Path(base_str)
    out_root = Path(out_root_str)
    
    parts = img_path.relative_to(base).parts
    
    # if len(parts)<3:
    #     return

    split , category = parts[0], parts[1]
    print(f"split{split}, category:{category}")
    out_dir = out_root / split / category
    out_dir.mkdir(parents=True, exist_ok=True)
    
    img = cv2.imread(str(img_path))
    if img is None:
        return
    
    with mp.solutions.pose.Pose(static_image_mode=True) as pose:
        results = pose.process(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        
    keypoints = []
    
    if results.pose_landmarks:
        for lmk in results.pose_landmarks.landmark:
            keypoints.append({
                "x": round(lmk.x, 5),
                "y": round(lmk.y, 5),
                "z": round(lmk.z, 5),
                "visibility": round(lmk.visibility, 5)
            })
                
    # save JSON alongside the image in the same split/category
    out_path = out_dir / f"{img_path.stem}_pose.json"
    with open(out_path, "w") as f:
        json.dump({"file": img_path.name, "landmarks": keypoints}, f, indent=2)


def label_all_poses_parallel(data_name="DeepFashion", max_workers=8):
    base = C.CLEAN_DIR / data_name

    out_root = C.POSE_DIR / data_name
    
    img_paths = []
    for ext in ("*.jpg","*.jpeg","*.png"):
        img_paths += [str(p) for p in base.rglob(ext)]
    
    print(f"==> Found {len(img_paths)} images in {data_name}")
    
   # Launch worker processes
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        futures = [
            executor.submit(process_single_image, p, str(base), str(out_root))
            for p in img_paths
        ]

        # 3) TQDM progress bar as each future completes
        for _ in tqdm(as_completed(futures),
                      total=len(futures),
                      desc=f"Labeling poses for {data_name}"):
            pass
            

label_all_poses_parallel()

==> Found 8706 images in DeepFashion


Labeling poses for DeepFashion: 100%|██████████| 8706/8706 [00:00<00:00, 15179.00it/s]
